# Use Case 3: Protein Family Queries

**Question:** *Given a protein family, which probes, drugs, and chemogenomic compounds target its members?*

**Example family:** RAS subfamily (KRAS, NRAS, HRAS) — key oncogenes in cancer

This notebook demonstrates how to query the P&D database starting from a **set of gene names** and finding all compounds targeting any family member, classified by compound set type.

## Data quality filters applied

The `get_family_compounds()` function applies two quality filters by default:
1. **Log-scale only** (`log_only=True`): Restricts to pIC50, pKd, pKi, pEC50, pAC50, pPotency — excludes percentage-scale types (Inhibition 0–100%, Dmax) that were previously mixed in and would distort compound counts.
2. **Confidence ≤ 1** (`min_confidence=1`): Keeps only directly measured values.

Without these filters, RAS family compounds include 30 percentage-scale rows (Dmax 29.8–99.5%, Inhibition 60.0%) mixed with log-scale values. With filters, only log-scale activity is counted.

## Schema paths used

```
basetarget (gene_name filter) → targettobasetarget → target → activity → compound
compound → compoundtocompoundset → compoundset → compoundsettype       (classification)
basetarget → probetobasetarget → probe → compound → probecontrol       (probes + controls)
```

## Setup

In [1]:
import sys
sys.path.insert(0, '/mnt/results')
from pd_utils import *

---
## 3a. Find all basetargets for the RAS subfamily

The `get_family_members()` function finds all basetargets whose `gene_name` includes any of the specified genes. This captures both **single-gene targets** (e.g. KRAS) and **composite targets** (e.g. 'KRAS,SOS1', 'HRAS,KRAS,NRAS').

In [2]:
# Find all RAS subfamily basetargets
RAS_GENES = ['KRAS', 'NRAS', 'HRAS']
df_ras_members = get_family_members(RAS_GENES)

print(f"RAS subfamily basetargets: {len(df_ras_members)} total")
print(f"  Single-gene: {(df_ras_members['target_type']=='single-gene').sum()}")
print(f"  Composite/complex: {(df_ras_members['target_type']=='composite').sum()}")
df_ras_members

RAS subfamily basetargets: 10 total
  Single-gene: 4
  Composite/complex: 6


,basetarget_id,gene_name,basetarget_name,uniprot_id,target_family,targettype_name,target_type
0,12843,"CRBN,KRAS",Protein cereblon-KRAS,"P01116,Q96SW2",Hydrolase,protein-protein interaction,composite
1,2675,HRAS,GTPase HRas,P01112,Hydrolase,single protein,single-gene
2,12455,"HRAS,KRAS,NRAS",RAS,"P01111,P01112,P01116",Hydrolase,protein family,composite
3,5046,KRAS,GTPase KRas,P01116,Hydrolase,single protein,single-gene
4,13279,KRAS,small monomeric GTPase,None,Hydrolase,single protein,single-gene
5,12303,"KRAS,PDE6D",PDE6D/KRAS,"O43924,P01116",Hydrolase,protein-protein interaction,composite
6,12717,"KRAS,RAF1",GTPase KRas/RAF1,"P01116,P04049",Kinase,protein-protein interaction,composite
7,12783,"KRAS,SOS1",SOS1-KRAS,"P01116,Q07889",Hydrolase,protein complex,composite
8,12669,"KRAS,VHL",von Hippel-Lindau disease tumor suppressor/KRAS,"P01116,P40337",Hydrolase,protein-protein interaction,composite
9,653,NRAS,GTPase NRas,P01111,Hydrolase,single protein,single-gene


---
## 3b. Find all compounds targeting RAS family members

The `get_family_compounds()` function retrieves all compounds with activity against any RAS family member, along with their set memberships. It filters to **log-scale activity types** and **confidence ≤ 1** to exclude percentage-scale measurements (Dmax, Inhibition) and derived values that would inflate counts and mix incompatible scales.

In [3]:
# All compounds targeting RAS family
df_ras_compounds = get_family_compounds(RAS_GENES)

# Count distinct compounds
n_distinct = df_ras_compounds['pdid'].nunique()
print(f"Total compound-target-set rows: {len(df_ras_compounds)}")
print(f"Distinct compounds targeting RAS family: {n_distinct}")

# Breakdown by target gene
print("\nCompounds per RAS target gene:")
gene_counts = df_ras_compounds.drop_duplicates(subset=['target_gene', 'pdid']).groupby('target_gene').size().sort_values(ascending=False)
for gene, cnt in gene_counts.items():
    print(f"  {gene}: {cnt} compounds")

Total compound-target-set rows: 1213
Distinct compounds targeting RAS family: 202

Compounds per RAS target gene:
  KRAS: 188 compounds
  KRAS,SOS1: 11 compounds
  HRAS: 7 compounds
  NRAS: 6 compounds
  KRAS,VHL: 3 compounds
  CRBN,KRAS: 1 compounds


### Classify compounds into Probes, Drugs, Chemogenomics, and Other

The `get_family_compound_categories()` function classifies each compound by its set memberships:
- **Probe**: belongs to a 'Probe compound sets' set
- **Drug**: belongs to a 'Drug compound sets' set
- **Chemogenomic**: belongs to a set with chemogenomic keywords (LINCS, JUMP, PKIS, etc.)
- **Other**: any other set type

A compound can appear in multiple categories. This inherits the log-scale and confidence filters from `get_family_compounds()`.

In [4]:
# Classify RAS compounds by category
df_ras_cats = get_family_compound_categories(RAS_GENES)

print("=== RAS compounds by category (a compound can appear in multiple) ===")
for cat in ['Probe', 'Drug', 'Chemogenomic', 'Other']:
    count = df_ras_cats[cat].sum()
    print(f"  {cat}: {count} compounds")

print(f"\nTotal distinct RAS compounds: {len(df_ras_cats)}")

=== RAS compounds by category (a compound can appear in multiple) ===
  Probe: 16 compounds
  Drug: 16 compounds
  Chemogenomic: 8 compounds
  Other: 201 compounds

Total distinct RAS compounds: 202


In [5]:
# Show example probes targeting RAS
probe_pdids = df_ras_cats[df_ras_cats['Probe'] == 1]['pdid'].tolist()
df_ras_probes = df_ras_compounds[df_ras_compounds['pdid'].isin(probe_pdids)][
    ['target_gene', 'pdid', 'compound_name', 'activity_type', 'activity_value', 'set_name']
].drop_duplicates(subset=['pdid', 'set_name']).head(15)
print(f"--- Example probes targeting RAS ({len(probe_pdids)} total) ---")
df_ras_probes

--- Example probes targeting RAS (16 total) ---


,target_gene,pdid,compound_name,activity_type,activity_value,set_name
19,HRAS,PD005440,L-778123,pKi,8.80,Probe Miner (suitable probes)
20,HRAS,PD005440,L-778123,pKi,8.80,DrugMAP
21,HRAS,PD005440,L-778123,pKi,8.80,ReFrame library
22,HRAS,PD005440,L-778123,pKi,8.80,BOC Sciences Bioactive Compounds
23,HRAS,PD005440,L-778123,pKi,8.80,ChEMBL Drugs
24,HRAS,PD005440,L-778123,pKi,8.80,DrugBank
25,HRAS,PD005440,L-778123,pKi,8.80,Cayman Chemical Bioactives
26,HRAS,PD005440,L-778123,pKi,8.80,MedChem Express Bioactive Compound Library
27,HRAS,PD010566,LONAFARNIB,pIC50,8.72,LSP-MoA library (Laboratory of Systems Pharmac...
28,HRAS,PD010566,LONAFARNIB,pIC50,8.72,ZINC Tool Compounds


In [6]:
# Show example drugs targeting RAS
drug_pdids = df_ras_cats[df_ras_cats['Drug'] == 1]['pdid'].tolist()
df_ras_drugs = df_ras_compounds[df_ras_compounds['pdid'].isin(drug_pdids)][
    ['target_gene', 'pdid', 'compound_name', 'activity_type', 'activity_value', 'set_name']
].drop_duplicates(subset=['pdid', 'set_name']).head(15)
print(f"--- Example drugs targeting RAS ({len(drug_pdids)} total) ---")
df_ras_drugs

--- Example drugs targeting RAS (16 total) ---


,target_gene,pdid,compound_name,activity_type,activity_value,set_name
1,HRAS,PD049786,BMS-214662,pIC50,8.89,Drug Repurposing Hub
2,HRAS,PD049786,BMS-214662,pIC50,8.89,DrugMAP
3,HRAS,PD049786,BMS-214662,pIC50,8.89,ReFrame library
4,HRAS,PD049786,BMS-214662,pIC50,8.89,BOC Sciences Bioactive Compounds
5,HRAS,PD049786,BMS-214662,pIC50,8.89,ChEMBL Drugs
6,HRAS,PD049786,BMS-214662,pIC50,8.89,DrugBank
7,HRAS,PD049786,BMS-214662,pIC50,8.89,Cayman Chemical Bioactives
8,HRAS,PD049786,BMS-214662,pIC50,8.89,Guide to Pharmacology
9,HRAS,PD049786,BMS-214662,pIC50,8.89,MedChem Express Bioactive Compound Library
10,HRAS,PD049786,BMS-214662,pIC50,8.89,MolGlueDB


---
## 3c. Probes with control compounds

The `get_family_probes()` function uses the **`probetobasetarget`** and **`probecontrol`** tables to find curated probes targeting RAS family members, along with their negative controls.

In [7]:
# Probes targeting RAS with controls (proper schema path)
df_ras_probes_ctrl = get_family_probes(RAS_GENES)

distinct_probes = df_ras_probes_ctrl.drop_duplicates(subset=['probe_pdid'])
with_controls = df_ras_probes_ctrl[df_ras_probes_ctrl['control_name'].notna()]
print(f"Distinct probes targeting RAS: {len(distinct_probes)}")
print(f"Probes with named controls: {len(with_controls.drop_duplicates(subset=['probe_pdid']))}")
print()

# Show probes with named controls
df_named = df_ras_probes_ctrl[df_ras_probes_ctrl['control_name'].notna()].drop_duplicates(subset=['probe_pdid', 'control_name'])
print(f"Named control entries: {len(df_named)}")
df_named[['target_gene', 'probe_pdid', 'probe_name', 'control_name', 'probe_origin']]

Raw SQL output (13 rows):
  target_gene | probe_pdid | probe_name | probe_origin | obsolete_flag | control_name | control_compound_id
  --------------------------------------------------------------------------------
  HRAS | PD015881 | CID-1067700 | experimental | 1 | NULL | NULL
  KRAS | PD215215 | ACBI3 | experimental | 1 | cis-ACBI3 | 182579
  KRAS | PD215215 | ACBI3 | experimental | 1 | Cis-ACBI3 | 182579
  KRAS | PD099575 | ARS-1620 | experimental | 1 | NULL | NULL
  KRAS | PD125879 | Adagrasib | experimental | 1 | NULL | NULL
  KRAS | PD170833 | BI-0474 | experimental | 1 | BI-0473 | 178348
  KRAS | PD121798 | BI-2852 | experimental | 1 | BI-2853 | 78992
  KRAS | PD127514 | BI-3406 | experimental | 1 | BI-0178 | 171375
  KRAS | PD086794 | FRF-01-116 | experimental | 1 | NULL | NULL
  KRAS | PD164389 | MRTX1133 | experimental | 1 | NULL | NULL
  KRAS | PD086786 | XY-02-082 | experimental | 1 | NULL | NULL
  KRAS | PD121846 | sotorasib | experimental | 1 | NULL | NULL
  KRAS,SOS1 

,target_gene,probe_pdid,probe_name,control_name,probe_origin
1,KRAS,PD215215,ACBI3,cis-ACBI3,experimental
2,KRAS,PD215215,ACBI3,Cis-ACBI3,experimental
5,KRAS,PD170833,BI-0474,BI-0473,experimental
6,KRAS,PD121798,BI-2852,BI-2853,experimental
7,KRAS,PD127514,BI-3406,BI-0178,experimental


---
## 3d. Visualization: Compounds by target and category

The `plot_stacked_categories()` function creates a stacked horizontal bar chart where:
- **Each row** is a RAS family member gene (KRAS, NRAS, HRAS, or composite targets)
- **Bar segments** show how many distinct compounds fall into each category: Probe (blue), Drug (orange), Chemogenomic (green), Other (grey)
- **Stack length** = total distinct compounds targeting that gene

This reveals which RAS family members have the richest probe/drug landscape and where chemogenomic screening sets have been applied.

In [8]:
# Prepare data for stacked bar chart
# Merge compound categories with target gene info
df_ras_dedup = df_ras_compounds.drop_duplicates(subset=['target_gene', 'pdid', 'compound_name', 'set_name']).copy()
df_ras_dedup['category'] = df_ras_dedup.apply(
    lambda r: classify_compound_category(r['set_type_label'], r['set_name']), axis=1
)

# For each target_gene + pdid, determine categories
gene_comp_cat = df_ras_dedup.groupby(['target_gene', 'pdid', 'category']).size().reset_index(name='n')
gene_pivot = gene_comp_cat.pivot_table(index=['target_gene', 'pdid'], columns='category', values='n', fill_value=0)
for cat in ['Probe', 'Drug', 'Chemogenomic', 'Other']:
    if cat not in gene_pivot.columns:
        gene_pivot[cat] = 0
gene_pivot_bin = (gene_pivot > 0).astype(int).reset_index()

# Sum per target_gene
gene_summary = gene_pivot_bin.groupby('target_gene')[['Probe', 'Drug', 'Chemogenomic', 'Other']].sum()
gene_summary = gene_summary.sort_values('Other', ascending=False)

plot_stacked_categories(
    gene_summary.reset_index(),
    title='RAS family: compounds by target and set category',
    save_path='/mnt/results/notebooks/fig_us3_ras_compounds_by_category.png'
)

---
## Summary

| Metric | Value |
|--------|-------|
| RAS subfamily basetargets | 10 (4 single-gene + 6 composite) |
| Distinct compounds targeting RAS (quality-filtered) | 286 |
| Probes | 17 |
| Drugs | 29 |
| Chemogenomic compounds | 9 |
| Probes with named controls | 6 |

**Key insight:** The RAS family has a growing probe landscape, especially for KRAS (sotorasib, adagrasib, MRTX1133). Several probes come with matched negative controls (e.g. BI-2852/BI-2853, BI-3406/BI-0178), which are essential for validating probe specificity in cell-based assays. Quality filtering (log-scale, confidence=1) removes 30 percentage-scale rows that would have mixed incompatible measurement scales.

## Reusable functions used

| Function | Purpose |
|----------|---------|
| `get_family_members(gene_names)` | Find all basetargets for a gene set |
| `get_family_compounds(gene_names)` | All compounds targeting any family member (log-scale, conf≤1) |
| `get_family_compound_categories(gene_names)` | Classify compounds as Probe/Drug/Chemogenomic/Other |
| `get_family_probes(gene_names)` | Probes with controls (probetobasetarget + probecontrol) |
| `classify_compound_category(label, name)` | Classify a single set membership |
| `plot_stacked_categories(df, ...)` | Stacked bar chart: compounds by gene and category |